# 12 — Figure panel revision fixes

This notebook regenerates selected manuscript figure panels requiring targeted visual or labeling corrections during revision.

The scope is limited to panel-level figure updates. The analytical criteria used to generate the panels are kept consistent with the original manuscript workflow unless explicitly noted.

Panels addressed:

- **Figure 2A**: revised title and terminology to reflect ranking by mean absolute transcriptional response.
- **Figure 3B**: revised consensus Hallmark enrichment bar plot with clearer `n_cells` labeling.
- **Figure 3C**: revised gene-level effects heatmap reported as top 10 genes, matching the displayed panel.

Expected outputs:

- `results/revision/figures/figure2A_revised.svg`
- `results/revision/figures/figure3B_revised.svg`
- `results/revision/figures/figure3C_revised.svg`

---

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parents[1].resolve()

DATA_EXPORTS_DIR = PROJECT_ROOT / "data" / "exports"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw_data"
ENRICHMENT_DIR = PROJECT_ROOT / "results" / "enrichment"
REVISION_FIGURE_DIR = PROJECT_ROOT / "results" / "revision" / "figures"

REVISION_FIGURE_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Input files
# ---------------------------------------------------------------------

INPUT_FILES = {
    "expression_matrix": DATA_EXPORTS_DIR / "expression_matrix_clean.parquet",
    "signature_metadata": DATA_EXPORTS_DIR / "signature_metadata_clean.csv",
    "gene_info": RAW_DATA_DIR / "geneinfo_beta.txt",
}

missing_inputs = {
    name: path
    for name, path in INPUT_FILES.items()
    if not path.exists()
}

if missing_inputs:
    missing_report = "\n".join(
        f"- {name}: {path.relative_to(PROJECT_ROOT)}"
        for name, path in missing_inputs.items()
    )
    raise FileNotFoundError(
        "Missing required input file(s):\n"
        f"{missing_report}"
    )

expected_cell_lines = ["A549", "HA1E", "MCF7", "PC3", "U2OS"]

hallmark_preview_files = {
    cell_id: ENRICHMENT_DIR / f"cell_Hallmarks_{cell_id}.preview.csv"
    for cell_id in expected_cell_lines
}

missing_hallmark_previews = {
    cell_id: path
    for cell_id, path in hallmark_preview_files.items()
    if not path.exists()
}

if missing_hallmark_previews:
    missing_report = "\n".join(
        f"- {cell_id}: {path.relative_to(PROJECT_ROOT)}"
        for cell_id, path in missing_hallmark_previews.items()
    )
    raise FileNotFoundError(
        "Missing required Hallmark preview file(s):\n"
        f"{missing_report}"
    )


# ---------------------------------------------------------------------
# Output files
# ---------------------------------------------------------------------

OUTPUT_FILES = {
    "figure2A_revised": REVISION_FIGURE_DIR / "figure2A_revised.svg",
    "figure3B_revised": REVISION_FIGURE_DIR / "figure3B_revised.svg",
    "figure3C_revised": REVISION_FIGURE_DIR / "figure3C_revised.svg",
}


# ---------------------------------------------------------------------
# Lightweight input loading and validation
# ---------------------------------------------------------------------

expression_matrix = pd.read_parquet(INPUT_FILES["expression_matrix"])
signature_metadata = pd.read_csv(INPUT_FILES["signature_metadata"])
gene_info = pd.read_csv(INPUT_FILES["gene_info"], sep="\t", dtype=str)

expression_matrix.index = expression_matrix.index.astype(str)
expression_matrix.columns = expression_matrix.columns.astype(str)

required_metadata_columns = {
    "sig_id",
    "cell_id",
}

required_geneinfo_columns = {
    "gene_id",
    "gene_symbol",
}

missing_metadata_columns = required_metadata_columns - set(signature_metadata.columns)
missing_geneinfo_columns = required_geneinfo_columns - set(gene_info.columns)

if missing_metadata_columns:
    raise ValueError(
        "Missing required column(s) in signature metadata: "
        f"{sorted(missing_metadata_columns)}"
    )

if missing_geneinfo_columns:
    raise ValueError(
        "Missing required column(s) in gene info: "
        f"{sorted(missing_geneinfo_columns)}"
    )

signature_metadata = signature_metadata.copy()
signature_metadata["sig_id"] = signature_metadata["sig_id"].astype(str)

expression_signature_ids = set(expression_matrix.columns)
metadata_signature_ids = set(signature_metadata["sig_id"])

missing_from_expression = metadata_signature_ids - expression_signature_ids
missing_from_metadata = expression_signature_ids - metadata_signature_ids

if missing_from_expression or missing_from_metadata:
    raise ValueError(
        "Expression matrix and signature metadata are not fully aligned by sig_id.\n"
        f"Metadata signatures missing from expression matrix: {len(missing_from_expression)}\n"
        f"Expression signatures missing from metadata: {len(missing_from_metadata)}"
    )

signature_metadata_aligned = (
    signature_metadata
    .set_index("sig_id")
    .loc[expression_matrix.columns]
)

observed_cell_lines = sorted(signature_metadata_aligned["cell_id"].unique())

if observed_cell_lines != expected_cell_lines:
    raise ValueError(
        "Unexpected cell-line coverage in signature metadata.\n"
        f"Expected: {expected_cell_lines}\n"
        f"Observed: {observed_cell_lines}"
    )


# ---------------------------------------------------------------------
# Reporting
# ---------------------------------------------------------------------

print("Project root:")
print(PROJECT_ROOT)

print("\nInput files:")
for name, path in INPUT_FILES.items():
    print(f"- {name}: {path.relative_to(PROJECT_ROOT)}")

print("\nHallmark preview files:")
for cell_id, path in hallmark_preview_files.items():
    print(f"- {cell_id}: {path.relative_to(PROJECT_ROOT)}")

print("\nOutput files:")
for name, path in OUTPUT_FILES.items():
    print(f"- {name}: {path.relative_to(PROJECT_ROOT)}")

print("\nLoaded inputs:")
print(f"- expression_matrix: {expression_matrix.shape[0]} genes x {expression_matrix.shape[1]} signatures")
print(f"- signature_metadata: {signature_metadata.shape[0]} rows x {signature_metadata.shape[1]} columns")
print(f"- gene_info: {gene_info.shape[0]} rows x {gene_info.shape[1]} columns")

print("\nAlignment checks:")
print(f"- expression columns match metadata sig_id values: {expression_signature_ids == metadata_signature_ids}")
print(f"- cell lines: {observed_cell_lines}")
print(f"- output directory exists: {REVISION_FIGURE_DIR.exists()}")

In [ ]:
# ---------------------------------------------------------------------
# Prepare source data for revised panels
# ---------------------------------------------------------------------

# Gene ID to gene symbol mapping
gene_info_clean = (
    gene_info.loc[:, ["gene_id", "gene_symbol"]]
    .dropna(subset=["gene_id"])
    .assign(gene_id=lambda df: df["gene_id"].astype(str))
    .drop_duplicates(subset="gene_id", keep="first")
)

gene_symbol_map = gene_info_clean.set_index("gene_id")["gene_symbol"]

In [ ]:
# ---------------------------------------------------------------------
# Figure 2A source data
# ---------------------------------------------------------------------
# Original criterion retained:
# rank genes by mean absolute LINCS Level 5 z-score across all signatures.

figure2A_data = (
    expression_matrix
    .abs()
    .mean(axis=1)
    .rename("mean_abs_zscore")
    .to_frame()
    .assign(
        gene_id=lambda df: df.index.astype(str),
        gene_symbol=lambda df: (
            df["gene_id"]
            .map(gene_symbol_map)
            .fillna(df["gene_id"])
        ),
    )
    .sort_values("mean_abs_zscore", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

In [ ]:
# ---------------------------------------------------------------------
# Effects by cell line for Figure 3C
# ---------------------------------------------------------------------
# Mean gene-level effect per cell line, matching the directed-results workflow.

effects_by_cell = pd.DataFrame(index=expression_matrix.index)

for cell_id in expected_cell_lines:
    cell_sig_ids = (
        signature_metadata_aligned
        .loc[signature_metadata_aligned["cell_id"] == cell_id]
        .index
        .tolist()
    )

    effects_by_cell[cell_id] = expression_matrix[cell_sig_ids].mean(axis=1)

gene_symbols_for_effects = (
    effects_by_cell
    .index
    .to_series(index=effects_by_cell.index)
    .map(gene_symbol_map)
    .fillna(effects_by_cell.index.to_series(index=effects_by_cell.index))
)

effects_by_cell_annotated = effects_by_cell.copy()
effects_by_cell_annotated["gene_symbol"] = gene_symbols_for_effects.values

# Figure 3C revised panel uses top 10 genes, matching the displayed panel.
figure3C_gene_ids = figure2A_data["gene_id"].head(10).tolist()

figure3C_data = (
    effects_by_cell_annotated
    .loc[figure3C_gene_ids]
    .set_index("gene_symbol")
    .loc[:, expected_cell_lines]
)

In [ ]:
# ---------------------------------------------------------------------
# Figure 3B source data
# ---------------------------------------------------------------------
# Reconstruct consensus Hallmark enrichment from existing per-cell preview files.

required_enrichment_columns = {
    "term",
    "ES",
    "fdr_bh",
}

hallmark_by_cell = {}

for cell_id, path in hallmark_preview_files.items():
    table = pd.read_csv(path)

    missing_columns = required_enrichment_columns - set(table.columns)
    if missing_columns:
        raise ValueError(
            f"Missing required column(s) in {path.relative_to(PROJECT_ROOT)}: "
            f"{sorted(missing_columns)}"
        )

    hallmark_by_cell[cell_id] = table.copy()

hallmark_significant_long = []

for cell_id, table in hallmark_by_cell.items():
    significant = table.loc[
        table["fdr_bh"] < 0.05,
        ["term", "ES", "fdr_bh"],
    ].copy()

    significant["cell_id"] = cell_id
    hallmark_significant_long.append(significant)

hallmark_significant_long = pd.concat(
    hallmark_significant_long,
    ignore_index=True,
)

consensus_hallmark_data = (
    hallmark_significant_long
    .groupby("term", as_index=False)
    .agg(
        n_cells=("cell_id", "nunique"),
        mean_ES=("ES", "mean"),
    )
    .sort_values(["n_cells", "mean_ES"], ascending=[False, False])
    .reset_index(drop=True)
)

figure3B_data = consensus_hallmark_data.head(10).copy()

figure3B_data["term_label"] = (
    figure3B_data["term"]
    .str.replace("HALLMARK_", "", regex=False)
    .str.replace("_", " ", regex=False)
)

In [ ]:
# ---------------------------------------------------------------------
# Reporting
# ---------------------------------------------------------------------

print("Figure 2A source data:")
print(f"- rows: {figure2A_data.shape[0]}")
print(f"- ranking metric: mean_abs_zscore")
print(f"- top gene: {figure2A_data.loc[0, 'gene_symbol']}")

print("\nFigure 3C source data:")
print(f"- shape: {figure3C_data.shape[0]} genes x {figure3C_data.shape[1]} cell lines")
print(f"- displayed genes: {list(figure3C_data.index)}")

print("\nFigure 3B source data:")
print(f"- significant Hallmark rows used: {hallmark_significant_long.shape[0]}")
print(f"- consensus Hallmark terms: {consensus_hallmark_data.shape[0]}")
print(f"- displayed terms: {figure3B_data.shape[0]}")

print("\nFigure 3B displayed terms:")
print(
    figure3B_data[
        ["term", "term_label", "n_cells", "mean_ES"]
    ].to_string(index=False)
)

In [ ]:
# ---------------------------------------------------------------------
# Generate revised Figure 2A
# ---------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(5, 4.5), constrained_layout=True)

sns.barplot(
    data=figure2A_data,
    x="mean_abs_zscore",
    y="gene_symbol",
    hue="gene_symbol",
    palette="viridis",
    dodge=False,
    legend=False,
    ax=ax,
)

ax.set_xlabel("Mean |z-score| across all signatures", fontsize=10)
ax.set_ylabel("Gene symbol", fontsize=10)
ax.set_title(
    "Top 20 genes by mean absolute transcriptional response",
    fontsize=10,
)

ax.tick_params(axis="y", labelsize=8)
ax.tick_params(axis="x", labelsize=8)

fig.savefig(
    OUTPUT_FILES["figure2A_revised"],
    bbox_inches="tight",
)

plt.show()

print("Saved revised Figure 2A:")
print(f"- {OUTPUT_FILES['figure2A_revised'].relative_to(PROJECT_ROOT)}")

In [ ]:
# ---------------------------------------------------------------------
# Generate revised Figure 3B
# ---------------------------------------------------------------------

figure3B_plot = figure3B_data.copy()

fig, ax = plt.subplots(figsize=(4.2, 3.1), constrained_layout=True)

sns.barplot(
    data=figure3B_plot,
    x="mean_ES",
    y="term_label",
    color="#2f7fad",
    ax=ax,
)

ax.set_xlabel("Mean enrichment score across cell lines", fontsize=9)
ax.set_ylabel("")
ax.set_title(
    "Consensus Hallmark enrichment across cell lines",
    fontsize=10,
)

ax.tick_params(axis="x", labelsize=8)
ax.tick_params(axis="y", labelsize=8)

xmax = figure3B_plot["mean_ES"].max()
ax.set_xlim(0, xmax + 0.08)

for i, row in figure3B_plot.reset_index(drop=True).iterrows():
    ax.text(
        row["mean_ES"] + 0.012,
        i,
        f"n={int(row['n_cells'])}",
        va="center",
        ha="left",
        fontsize=8,
        clip_on=False,
    )

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.savefig(
    OUTPUT_FILES["figure3B_revised"],
    bbox_inches="tight",
)

plt.show()

print("Saved revised Figure 3B:")
print(f"- {OUTPUT_FILES['figure3B_revised'].relative_to(PROJECT_ROOT)}")

In [ ]:
# ---------------------------------------------------------------------
# Generate revised Figure 3C
# ---------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(3.2, 3.0), constrained_layout=True)

sns.heatmap(
    figure3C_data,
    cmap="vlag",
    center=0,
    linewidths=0.3,
    linecolor="white",
    cbar_kws={"label": "Gene-level effect size"},
    ax=ax,
)

ax.set_xlabel("Cell line", fontsize=9)
ax.set_ylabel("Gene symbol", fontsize=9)
ax.set_title(
    "Top 10 gene-level effects across cell lines",
    fontsize=10,
)

ax.tick_params(axis="x", labelsize=8, rotation=0)
ax.tick_params(axis="y", labelsize=8)

fig.savefig(
    OUTPUT_FILES["figure3C_revised"],
    bbox_inches="tight",
)

plt.show()

print("Saved revised Figure 3C:")
print(f"- {OUTPUT_FILES['figure3C_revised'].relative_to(PROJECT_ROOT)}")